In [17]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("Data Cleaning Curated 2")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '8g')
    .config('spark.executor.memory', '4g')
    .getOrCreate()
)
# Silence any WARN-level logs from this point forth.
spark.sparkContext.setLogLevel("ERROR")

In [18]:
path = "../../rea_data/analysis/vic_rentals_merged_2.parquet"

df = spark.read.parquet(path)

df.printSchema()

root
 |-- LGA: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- listing_id: string (nullable = true)
 |-- suburb: string (nullable = true)
 |-- weekly_rent: double (nullable = true)
 |-- date_listed: date (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- carspaces: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- feat_built_in_wardrobes: integer (nullable = true)
 |-- feat_heating: integer (nullable = true)
 |-- feat_dishwasher: integer (nullable = true)
 |-- feat_secure_parking: integer (nullable = true)
 |-- feat_internal_laundry: integer (nullable = true)
 |-- feat_air_conditioning: integer (nullable = true)
 |-- feat_balcony___deck: integer (nullable = true)
 |-- feat_gas: integer (nullable = true)
 |-- feat_floorboards: integer (nullable = true)
 |-- feat_bath: integer (nullable = true)
 |-- feat_intercom: integer (

<br>
<br>
<br>

### Merging With Population Data

Income data is granularised by SA2 regions, so in order to merge with our curated rental data, we must use the `lat` and `lon` information to calculate which SA2 a given listing lies within. Then merging on SA2 becomes very simple. However, from our previous merge, we already know which SA2 region each listing is in, so the rest becomes simple.

In [19]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

SA2_FILE = "../../external_data/SA2_2021_AUST_SHP_GDA2020/SA2_2021_AUST_GDA2020.shp"
INCOME_FILE = "../../external_data/abs_income_sa2_vic_clean.csv"
INCOME_COL = "median_equivalised_total_household_income_weekly_"

Now, lets the census income data, keeping only the SA2 code, census year and median weekly household income. The SA2 code is cast to text so it matches the listings' SA2_code column.

In [20]:
# Income: keep code, year and income; tidy the names
income = (
    spark.read.csv(INCOME_FILE, header=True, inferSchema=True)
    .select(
        F.col("code").cast("string").alias("SA2_code"),
        F.col("year").alias("census_year"),
        F.col(INCOME_COL).alias("median_weekly_income"),
    )
)
income.show(5)

+---------+-----------+--------------------+
| SA2_code|census_year|median_weekly_income|
+---------+-----------+--------------------+
|201011001|       2011|               786.0|
|201011001|       2016|               913.0|
|201011001|       2021|              1113.0|
|201011002|       2011|               762.0|
|201011002|       2016|               903.0|
+---------+-----------+--------------------+
only showing top 5 rows


Income is only measured in census years, so lets assign each listing the most recent census before it: 2011 for 2012–2016, 2016 for 2017–2021, and 2021 for 2022 onwards. The table confirms which listing years map to each census.

In [21]:
# Census is in August, so census C first applies to Year C + 1.
# Use the most recent census before each listing.
df = df.withColumn(
    "census_year",
    F.when(F.col("Year") <= 2016, 2011)
    .when(F.col("Year") <= 2021, 2016)
    .otherwise(2021),
)

df.groupBy("census_year").agg(
    F.min("Year").alias("min_year"),
    F.max("Year").alias("max_year"),
    F.count("*").alias("listings"),
).orderBy("census_year").show()

+-----------+--------+--------+--------+
|census_year|min_year|max_year|listings|
+-----------+--------+--------+--------+
|       2011|    2012|    2016|      27|
|       2016|    2017|    2021|      30|
|       2021|    2022|    2026|   12259|
+-----------+--------+--------+--------+



This adds each listing's median weekly household income by matching its SA2 and census year, keeping every listing. It then checks that the row count didn't change and counts listings without an income value.

In [22]:
# Count rows first, so we can check the join doesn't duplicate any
rows_before = df.count()

# Add each listing's income, matched on its SA2 and census year
df = df.join(income, on=["SA2_code", "census_year"], how="left")

# Row counts should match; any unmatched listings have no income
print("Rows before:", rows_before, "| after:", df.count())
missing = df.filter(F.col("median_weekly_income").isNull()).count()
print(missing, "listings with no income")

Rows before: 12316 | after: 12316
0 listings with no income


Lets just do a few final sanity checks to make sure the merge has been done correctly and the numbers make sense.

In [23]:
# 1. Missing values
missing = df.filter(F.col("median_weekly_income").isNull()).count()
print(missing, "listings with no income")

# 2. Range: the source file runs from $337 to $3,500 a week
df.select("median_weekly_income").summary(
    "min", "25%", "50%", "75%", "max"
).show()

# 3. Outliers: $3,000+ only occurs in Royal Botanic Gardens
high = df.filter(F.col("median_weekly_income") >= 3000).count()
print(high, "listings with income >= $3,000")

# 4. Trend: median income should rise from census to census
df.groupBy("census_year").agg(
    F.round(F.avg("median_weekly_income"), 0).alias("avg_income"),
    F.count("*").alias("listings"),
).orderBy("census_year").show()

# 5. Type: should be double, not string
df.select("median_weekly_income").printSchema()

0 listings with no income
+-------+--------------------+
|summary|median_weekly_income|
+-------+--------------------+
|    min|               436.0|
|    25%|               966.0|
|    50%|              1117.0|
|    75%|              1369.0|
|    max|              1921.0|
+-------+--------------------+

0 listings with income >= $3,000
+-----------+----------+--------+
|census_year|avg_income|listings|
+-----------+----------+--------+
|       2011|     849.0|      27|
|       2016|     959.0|      30|
|       2021|    1171.0|   12259|
+-----------+----------+--------+

root
 |-- median_weekly_income: double (nullable = true)



<br>
<br>
<br>

### Saving To Disc

Before we save to disc, lets just drop all the columns we no longer need.

In [24]:
df.printSchema()

root
 |-- SA2_code: string (nullable = true)
 |-- census_year: integer (nullable = false)
 |-- LGA: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- listing_id: string (nullable = true)
 |-- suburb: string (nullable = true)
 |-- weekly_rent: double (nullable = true)
 |-- date_listed: date (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- carspaces: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- feat_built_in_wardrobes: integer (nullable = true)
 |-- feat_heating: integer (nullable = true)
 |-- feat_dishwasher: integer (nullable = true)
 |-- feat_secure_parking: integer (nullable = true)
 |-- feat_internal_laundry: integer (nullable = true)
 |-- feat_air_conditioning: integer (nullable = true)
 |-- feat_balcony___deck: integer (nullable = true)
 |-- feat_gas: integer (nullable = true)
 |-- feat_floorboards: integer (nul

In [25]:
df = df.drop("census_year","SA2_code", "LGA", "pop_year")

Finally, save to disc

In [26]:
n_rows = df.count()
n_cols = len(df.columns)
print(f"Dataset shape: {n_rows:,} rows x {n_cols} columns")

Dataset shape: 12,316 rows x 69 columns


In [27]:
OUTPUT_PATH = "../../rea_data/analysis/vic_rentals_merged_3.parquet"

df.write.mode("overwrite").parquet(OUTPUT_PATH)
print("Saved to", OUTPUT_PATH)

Saved to ../../rea_data/analysis/vic_rentals_merged_3.parquet
